# Dual-basis (vertex-token) gridinv on Colab — L=4 tuning + primal/dual A/B

**Part 1 — tuning, one fully-explicit cell per run:** every architectural and training
hyperparameter is spelled out in each cell — noninv block (depth `n_noninv` × channels
`noninv_channels`, stencil `radius_edge`), invariant block (widths `inv_hidden`, depth =
its length, grid-conv `kernel_size`), optimizer (`n_iter`, `dt`→`lr_min` cosine,
`diag_shift`, `qgt`) and sampling (`n_samples`, `n_chains`, `n_sweeps`, `chunk_size`).
All runs are anchored against a primal production-config reference and **log live to
wandb.ai** (per-step energy/spread/Vscore, full config, timing, weight artifacts).

**Kernel/footprint cheat sheet:**
- `kernel_size` = the INVARIANT block's cubic nn.Conv3D extent on the token grid
  (1…L; L = full span). Primal tokens: (L,L,L)×3C channels; dual: (L,L,L)×C.
- `radius_edge` = the NONINV block's geometry-exact stencil radius on the edge
  lattice: 1.05 → 15 taps (self + 8 perpendicular NN + 6 same-orientation next-NN);
  0.9 → 9 taps (self + 8, the star-adjacent footprint); ~1.5 → next shell.
- inv depth = `len(inv_hidden)` hidden convs + 1 readout conv; noninv depth = `n_noninv`.

**Dual-net constraints:** star product is per-channel ⇒ invariant input width == C
(primal gets 3C from orientation folding) ⇒ ~3.8× fewer params at matched widths.
L=4 counts (primal k3 inv"2 2 2" = 5319): dual k4 inv"4 4 4" = 4265 (0.80×),
C=8 + inv"2 2 2" = 4959 (0.93×), k4 inv"8 4 2" = 5675 (1.07×), inv"8 8" = 7597 (1.43×).
The 6-edge star product is sharper than the primal 4-product — if energy staircases or
`[guard]` fires, raise `diag_shift`. Under OBC only (L−2)³ stars are full (8/64 at L=4).

**Part 2 — the North Star:** matched-param primal vs dual at 2 hx-cut points + 1 hz
control, judged on final E (variational), Vscore, stability; ⟨A_v⟩/⟨B_p⟩/⟨M_z⟩ must
agree between arms (keys are physical — disagreement = conjugation bug).

Runtime: L=4 ≈ 4 s/step on A100 (~15 min per 200-iter run); T4/L4: set
`chunk_size=1024` and expect 2–4× slower.


In [ ]:
# ============================== §1 CONFIG ====================================
REPO_URL = "https://github.com/SanzharBissenali/ThreeD_TC.git"
BRANCH   = "feat/dual-basis"

OUT_ROOT      = "outputs/dual_basis_colab"   # per-run JSON/mpack/curve/log land here
SKIP_EXISTING = True                         # finished {name}.json -> skip (resume-safe)

WANDB         = True                         # live logging to wandb.ai
WANDB_PROJECT = "approx-sym-3D-TC"
WANDB_ENTITY  = None                         # None -> train.py default entity


In [ ]:
# ============================== §2 SETUP =====================================
# Pin the EXACT production stack (nersc/setup_conda_gpu.sh == local .venv): newest
# PyPI jax/netket/wandb break or crawl (n_sweeps removal, QGT NotImplementedError,
# slow wandb-core). If pip asks to restart the runtime (numpy), do it, re-run this cell.
import os, sys, json, glob, subprocess
%cd /content
if not os.path.isdir("repo"):
    !git clone --branch $BRANCH $REPO_URL repo
else:
    !cd repo && git fetch origin $BRANCH && git checkout $BRANCH && git pull
%pip -q install jax==0.5.2 jaxlib==0.5.1 "jax-cuda12-plugin[with-cuda]==0.5.1" \
    jax-cuda12-pjrt==0.5.1 netket==3.16.1.post1 flax==0.10.4 optax \
    "numpy==2.1.3" "scipy==1.15.2" wandb==0.27.2
%cd /content/repo

import jax
print("devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), \
    "no GPU — Runtime > Change runtime type > GPU (and re-run this cell)"

# W&B auth: API KEY in the environment, NEVER wandb.login()/import wandb in the
# notebook process — login boots a shared wandb-core service (WANDB_SERVICE) that
# training subprocesses stall attaching to. With the key in the env each run
# self-authenticates and starts its own service in seconds.
# Preferred: Colab sidebar > key icon (Secrets) > WANDB_API_KEY, notebook access ON.
if WANDB and "WANDB_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
        print("W&B key loaded from Colab secret")
    except Exception:
        from getpass import getpass
        os.environ["WANDB_API_KEY"] = getpass("wandb.ai API key: ")
os.environ.pop("WANDB_SERVICE", None)   # in case a login ran earlier this session


In [ ]:
# ============================== §3 helpers ===================================
import numpy as np

def run(name, *, dual, L, bc, hx, hz, seed,
        n_noninv=2, noninv_channels=4, radius_edge=1.05,  # noninv (edge) block
        inv_hidden="4 4", kernel_size=None,               # invariant block (None -> L-1)
        n_iter, dt, lr_min, diag_shift, qgt,             # optimizer
        n_samples, n_chains, n_sweeps, chunk_size,       # sampling
        noninv_hidden=None,   # per-layer noninv widths, e.g. "1 2 4" (spins -> 1
                              # -> 2 -> 4 -> Wilson); overrides n_noninv/noninv_channels
        arch="ToricCNN_gridinv",  # "GeoCNN" = symmetry-unaware control (edge convs,
        cnn_hidden=None,          #  no Wilson product); takes cnn_hidden="4 4 4"
        group=None, subdir="tune", **extra):
    """One independent training run (fresh subprocess -> fresh JAX memory).
    EVERY hyperparameter is an explicit argument — nothing hidden. Streams all
    output (per-step E/Vscore, [t] timing, [guard], wandb) into the cell and a
    sibling .log. Skips finished runs (SKIP_EXISTING keys on {name}.json)."""
    out_dir = f"{OUT_ROOT}/{subdir}_L{L}"
    jp = f"{out_dir}/{name}.json"
    if SKIP_EXISTING and os.path.exists(jp):
        print(f"[skip] {name} (done)"); return jp
    if kernel_size is None: kernel_size = L - 1      # house default: k = L-1
    cmd = [sys.executable, "-u", "-m", "Three_TC.train",
           "--L", str(L), "--bc", bc, "--arch", arch,
           "--hx", str(hx), "--hz", str(hz), "--seed", str(seed),
           "--n_iter", str(n_iter), "--dt", str(dt), "--lr_min", str(lr_min),
           "--diag_shift", str(diag_shift), "--qgt", qgt,
           "--n_samples", str(n_samples), "--n_chains", str(n_chains),
           "--n_sweeps", str(n_sweeps), "--chunk_size", str(chunk_size),
           "--radius_edge", str(radius_edge),
           "--out_dir", out_dir, "--name", name, "--resume",
           "--wandb_project", WANDB_PROJECT,
           "--wandb_group", group or f"{subdir}_L{L}"]
    if arch == "GeoCNN":
        cmd += ["--cnn_hidden", *str(cnn_hidden).split()]
    else:
        cmd += ["--n_noninv", str(n_noninv), "--noninv_channels", str(noninv_channels),
                "--inv_hidden", *str(inv_hidden).split(),
                "--kernel_size", str(kernel_size)]
        if noninv_hidden: cmd += ["--noninv_hidden", *str(noninv_hidden).split()]
    if dual:            cmd += ["--dual_basis"]
    if not WANDB:       cmd += ["--no_wandb"]
    if WANDB_ENTITY:    cmd += ["--wandb_entity", WANDB_ENTITY]
    for k, v in extra.items():
        cmd += [f"--{k}", str(v)]
    os.makedirs(out_dir, exist_ok=True)
    env = {k: v for k, v in os.environ.items() if k != "WANDB_SERVICE"}
    with open(f"{out_dir}/{name}.log", "w") as lg:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, env=env)
        for line in p.stdout:                 # stream EVERYTHING
            lg.write(line)
            print(line, end="", flush=True)
    if p.wait() != 0:
        print(f"NONZERO EXIT ({p.returncode}) — see {out_dir}/{name}.log")
    return jp

def n_params(**cfg):
    """Cheap parameter count: build the ansatz + init once, no training."""
    from Three_TC.builders import with_defaults, build_geometry, build_model
    c = with_defaults(dict(cfg)); geo = build_geometry(c)
    p = build_model(c, geo).init(jax.random.PRNGKey(0), np.ones((1, geo.N)))
    return sum(int(np.prod(np.shape(l))) for l in jax.tree_util.tree_leaves(p))

def load_all(subdir, L):
    docs = []
    for jp in sorted(glob.glob(f"{OUT_ROOT}/{subdir}_L{L}/*.json")):
        if jp.endswith(".curve.json"): continue
        with open(jp) as f: d = json.load(f)
        log = jp[:-5] + ".log"
        d["n_rollbacks"] = (open(log).read().count("[guard]") if os.path.exists(log) else 0)
        docs.append(d)
    return docs

def row(d):
    o, c = d["observables"], d["config"]
    sp = np.asarray(d["curve"]["energy_spread"], float)
    late = sp[-max(1, len(sp)//5):]
    return dict(name=d["name"], dual=bool(c.get("dual_basis")), E=o["E0"], E_err=o["E_err"],
                Vscore=o["Vscore"], A_v=o["A_v_mean"], B_p=o["B_p_mean"], M_z=o["sz_mean"],
                spread_late=float(np.median(late)),
                spread_spike=float(np.max(sp) / (np.median(sp) + 1e-30)),
                rollbacks=d["n_rollbacks"], diverged=bool(d.get("diverged")),
                n_params=c.get("n_params"),
                s_per_step=d["runtime_s"] / max(1, len(sp)))


## Part 1 — dual-net tuning at L=4, one run per cell

All runs at the same hx-dominated point (hx=0.7, hz=0.2, near the L=4 transition; the
dual net's diagonal regime). Edit any knob and re-run a single cell — but bump the
`name` when you change knobs, so runs never share a checkpoint or a W&B id.
Everything lands in W&B group `tune_L4`.


In [ ]:
# --- anchor: PRIMAL production config (5319 params) ---------------------------
run("primal_ref", dual=False,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,   # noninv: 2 layers, 15-tap
    inv_hidden="2 2 2", kernel_size=3,                 # inv: 3 convs (x3 orient) + readout
    L=4, bc="OBC", hx=0.7, hz=0.2, seed=0,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048)


In [ ]:
# --- dual A: full-span kernel, matched-ish capacity (4265 params, 0.80x) -------
run("dual_k4_inv444", dual=True,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="4 4 4", kernel_size=4,                 # k=4 = full span at L=4
    L=4, bc="OBC", hx=0.7, hz=0.2, seed=0,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048)


In [ ]:
# --- dual B: smaller kernel — is depth-stacked receptive field enough? (2341) --
run("dual_k3_inv444", dual=True,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="4 4 4", kernel_size=3,
    L=4, bc="OBC", hx=0.7, hz=0.2, seed=0,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048)


In [ ]:
# --- dual C: capacity floor — primal-matched WIDTHS, not params (~1400, 0.27x) --
run("dual_k4_inv222", dual=True,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="2 2 2", kernel_size=4,
    L=4, bc="OBC", hx=0.7, hz=0.2, seed=0,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048)


In [ ]:
# --- dual D: wide + shallow (7597 params, 1.43x) --------------------------------
run("dual_k4_inv88", dual=True,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="8 8", kernel_size=4,
    L=4, bc="OBC", hx=0.7, hz=0.2, seed=0,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048)


In [ ]:
# --- dual E: stiffer SR (the 6-edge star product can make gradients spiky) ------
run("dual_k4_inv444_ds5e3", dual=True,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="4 4 4", kernel_size=4,
    L=4, bc="OBC", hx=0.7, hz=0.2, seed=0,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=5e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048)


In [ ]:
# --- dual F: hotter learning rate -----------------------------------------------
run("dual_k4_inv444_dt02", dual=True,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="4 4 4", kernel_size=4,
    L=4, bc="OBC", hx=0.7, hz=0.2, seed=0,
    n_iter=200, dt=0.02, lr_min=0.002, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048)


In [ ]:
# --- dual G: capacity in the TOKENS — C=8, narrow inv (4959 params, 0.93x) -------
# star product is per-channel: invariant input width == noninv_channels (primal
# gets 3x from orientation folding); raising C restores token diversity instead
# of conv width. Closest param match to the primal anchor.
run("dual_k4_inv222_c8", dual=True,
    n_noninv=2, noninv_channels=8, radius_edge=1.05,
    inv_hidden="2 2 2", kernel_size=4,
    L=4, bc="OBC", hx=0.7, hz=0.2, seed=0,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048)


In [ ]:
# --- dual H: star-adjacent noninv footprint (radius 0.9 -> 9-tap: self + 8 perp) --
# the tightest 'token-aligned' dressing; cheaper noninv block (924 -> 564 params)
run("dual_k4_inv444_r09", dual=True,
    n_noninv=2, noninv_channels=4, radius_edge=0.9,
    inv_hidden="4 4 4", kernel_size=4,
    L=4, bc="OBC", hx=0.7, hz=0.2, seed=0,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048)


In [ ]:
# --- dual I: channel PYRAMID through the whole net -------------------------------
# noninv widens toward the Wilson product (spins -> 1 -> 2 -> 4), inv tapers to
# the readout (4 -> 4 -> 2 -> 1). noninv_hidden overrides n_noninv/noninv_channels
# (still listed for the record). Identity init is width-safe: channel 0 stays an
# exact spin pass-through at every widening, so the warm start is unchanged.
run("dual_pyramid_124_442", dual=True,
    n_noninv=3, noninv_channels=4, radius_edge=1.05,
    noninv_hidden="1 2 4",
    inv_hidden="4 4 2", kernel_size=4,
    L=4, bc="OBC", hx=0.7, hz=0.2, seed=0,
    n_iter=200, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048)


In [ ]:
# ============================== Part 1 analysis ==============================
import matplotlib.pyplot as plt
import pandas as pd
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})

docs = load_all("tune", 4)
tab = pd.DataFrame([row(d) for d in docs]).sort_values("E")
display(tab.round(6))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for d in docs:
    c, ref = d["curve"], d["name"] == "primal_ref"
    kw = dict(color="k", ls="--", lw=1.6, zorder=5) if ref else dict(lw=1.2)
    ax[0].plot(c["step"], c["energy"], label=d["name"], **kw)
    ax[1].semilogy(c["step"], c["energy_spread"], **kw)
ax[0].set(xlabel="step", ylabel="E", title="dual tuning vs primal (--k): L=4, hx=0.7, hz=0.2")
ax[1].set(xlabel="step", ylabel="energy spread  $\\sqrt{Var[H]}$", title="stability")
ax[0].legend(fontsize=7, loc="upper left")
fig.tight_layout()
# -> carry the winner's settings into the dual arm of Part 2.


## Part 2 — primal vs dual A/B (the North Star)

Same points, same seeds, matched parameter budget (check below; widen the dual's
`inv_hidden`/`noninv_channels` until the ratio is ~0.9–1.1). One cell per field point,
both arms × 2 seeds, every hyperparameter explicit — update the dual arm with the
Part-1 winner's settings before launching. W&B group `ab_L4`.


In [ ]:
# ============================== param match ==================================
common = dict(L=4, bc="OBC", arch="ToricCNN_gridinv", n_noninv=2, radius_edge=1.05)
np_primal = n_params(**common, noninv_channels=4, kernel_size=3, inv_hidden=[2, 2, 2])
np_dual   = n_params(**common, noninv_channels=4, dual_basis=True, kernel_size=4,
                     inv_hidden=[4, 4, 4])
print(f"n_params  primal={np_primal}  dual={np_dual}  ratio={np_dual/np_primal:.2f}")


In [ ]:
# --- point: hx=0.4, hz=0.2 (hx cut, inside topological) ---
for seed in (0, 1):
    run(f"primal_hx0.4_hz0.2_s{seed}", dual=False,
        n_noninv=2, noninv_channels=4, radius_edge=1.05,
        inv_hidden="2 2 2", kernel_size=3,
        L=4, bc="OBC", hx=0.4, hz=0.2, seed=seed,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
        n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048, subdir="ab")
    run(f"dual_hx0.4_hz0.2_s{seed}", dual=True,
        n_noninv=2, noninv_channels=4, radius_edge=1.05,
        inv_hidden="4 4 4", kernel_size=4,
        L=4, bc="OBC", hx=0.4, hz=0.2, seed=seed,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
        n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048, subdir="ab")


In [ ]:
# --- point: hx=0.7, hz=0.2 (hx cut, near the L=4 transition) ---
for seed in (0, 1):
    run(f"primal_hx0.7_hz0.2_s{seed}", dual=False,
        n_noninv=2, noninv_channels=4, radius_edge=1.05,
        inv_hidden="2 2 2", kernel_size=3,
        L=4, bc="OBC", hx=0.7, hz=0.2, seed=seed,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
        n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048, subdir="ab")
    run(f"dual_hx0.7_hz0.2_s{seed}", dual=True,
        n_noninv=2, noninv_channels=4, radius_edge=1.05,
        inv_hidden="4 4 4", kernel_size=4,
        L=4, bc="OBC", hx=0.7, hz=0.2, seed=seed,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
        n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048, subdir="ab")


In [ ]:
# --- point: hx=0.2, hz=0.36 (hz CONTROL — the primal's diagonal regime) ---
for seed in (0, 1):
    run(f"primal_hx0.2_hz0.36_s{seed}", dual=False,
        n_noninv=2, noninv_channels=4, radius_edge=1.05,
        inv_hidden="2 2 2", kernel_size=3,
        L=4, bc="OBC", hx=0.2, hz=0.36, seed=seed,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
        n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048, subdir="ab")
    run(f"dual_hx0.2_hz0.36_s{seed}", dual=True,
        n_noninv=2, noninv_channels=4, radius_edge=1.05,
        inv_hidden="4 4 4", kernel_size=4,
        L=4, bc="OBC", hx=0.2, hz=0.36, seed=seed,
        n_iter=300, dt=0.01, lr_min=0.001, diag_shift=1e-3, qgt="dense",
        n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048, subdir="ab")


In [ ]:
# ============================== Part 2 analysis ==============================
docs = load_all("ab", 4)
def _pt(d):
    c = d["config"]; return f"hx{c['hx']}_hz{c['hz']}"
tab = pd.DataFrame([dict(point=_pt(d), arm=("dual" if d["config"].get("dual_basis")
                                            else "primal"), seed=d["config"]["seed"],
                         **row(d)) for d in docs])
display(tab.drop(columns=["name", "dual"]).round(6))

pts = sorted(tab.point.unique())
fig, axes = plt.subplots(2, len(pts), figsize=(4.2 * len(pts), 7), squeeze=False)
ARM = {"primal": dict(color="0.35"), "dual": dict(color=plt.cm.plasma(0.65))}
for j, p in enumerate(pts):
    for d in docs:
        if _pt(d) != p: continue
        a = "dual" if d["config"].get("dual_basis") else "primal"
        c = d["curve"]
        axes[0][j].plot(c["step"], c["energy"], "-", lw=1.2, alpha=0.9,
                        label=f"{a} s{d['config']['seed']}", **ARM[a])
        axes[1][j].semilogy(c["step"], c["energy_spread"], "-", lw=1.2, **ARM[a])
    axes[0][j].set(title=p, xlabel="step", ylabel="E" if j == 0 else None)
    axes[1][j].set(xlabel="step", ylabel="spread" if j == 0 else None)
    axes[0][j].legend(fontsize=7, loc="upper right")
fig.tight_layout()

v = (tab.groupby(["point", "arm"])
        .agg(E=("E", "mean"), Vscore=("Vscore", "mean"),
             rollbacks=("rollbacks", "sum"), diverged=("diverged", "any"))
        .reset_index())
print("\n=== verdict (lower E wins; check Vscore + stability alongside) ===")
display(v.round(6))
for p in pts:   # physical-consistency gate: arms measure the SAME observables
    sub = tab[tab.point == p].groupby("arm")[["A_v", "B_p", "M_z"]].mean()
    if len(sub) == 2 and (sub.diff().abs().iloc[-1] > 0.05).any():
        print(f"WARNING {p}: arms disagree on a physical observable\n{sub}")


## Part 3 — symmetry-aware vs symmetry-unaware (dual basis, L=4, hx=0.2 hz=0.2)

Both arms are **dual-basis** (same Hamiltonian, sampler, SR, sampling budget); the ONLY
difference is the star Wilson product. Removing it collapses the two-block structure:
features never reach the vertex grid, so every layer is a GeoConv3D edge conv = `GeoCNN`.

Param-matched pairs (k = L-1 = 3):

| pair | aware (gridinv) | params | unaware (GeoCNN) | params |
|---|---|---|---|---|
| A | C4 n2, inv "4 4", k3 | 1905 | cnn_hidden "4 4 4" | 1839 |
| B | C4 n2, inv "4 4 4", k3 | 2341 | cnn_hidden "4 4 4 4" | 2571 |

Judge by Vscore, E, late spread, rollbacks. GeoCNN caveat: radius-1.05 stencil reaches
~1 lattice unit/layer, so depth ~ receptive field (part of the claim, not unfairness).

In [ ]:
run("bench_aware_A_s0", dual=True, arch="ToricCNN_gridinv",
    L=4, bc="OBC", hx=0.2, hz=0.2, seed=0,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="4 4", kernel_size=3,               # k = L-1
    n_iter=200, dt=0.02, lr_min=0.002, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048,
    group="bench_L4", subdir="bench")

In [ ]:
run("bench_aware_A_s1", dual=True, arch="ToricCNN_gridinv",
    L=4, bc="OBC", hx=0.2, hz=0.2, seed=1,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="4 4", kernel_size=3,               # k = L-1
    n_iter=200, dt=0.02, lr_min=0.002, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048,
    group="bench_L4", subdir="bench")

In [ ]:
run("bench_unaware_A_s0", dual=True, arch="GeoCNN",           # symmetry-unaware control
    cnn_hidden="4 4 4",
    L=4, bc="OBC", hx=0.2, hz=0.2, seed=0,
    n_iter=200, dt=0.02, lr_min=0.002, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048,
    group="bench_L4", subdir="bench")

In [ ]:
run("bench_unaware_A_s1", dual=True, arch="GeoCNN",           # symmetry-unaware control
    cnn_hidden="4 4 4",
    L=4, bc="OBC", hx=0.2, hz=0.2, seed=1,
    n_iter=200, dt=0.02, lr_min=0.002, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048,
    group="bench_L4", subdir="bench")

In [ ]:
run("bench_aware_B_s0", dual=True, arch="ToricCNN_gridinv",
    L=4, bc="OBC", hx=0.2, hz=0.2, seed=0,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="4 4 4", kernel_size=3,               # k = L-1
    n_iter=200, dt=0.02, lr_min=0.002, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048,
    group="bench_L4", subdir="bench")

In [ ]:
run("bench_unaware_B_s0", dual=True, arch="GeoCNN",           # symmetry-unaware control
    cnn_hidden="4 4 4 4",
    L=4, bc="OBC", hx=0.2, hz=0.2, seed=0,
    n_iter=200, dt=0.02, lr_min=0.002, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=48, chunk_size=2048,
    group="bench_L4", subdir="bench")

In [ ]:
# Part 3 analysis — aware vs unaware table
import pandas as pd
rows = [row(d) for d in load_all("bench", 4)]
df = pd.DataFrame(rows).sort_values("Vscore")
df["arm"] = np.where(df["name"].str.contains("unaware"), "unaware", "aware")
cols = ["name", "arm", "n_params", "E", "E_err", "Vscore", "spread_late",
        "spread_spike", "rollbacks", "s_per_step", "A_v", "B_p", "M_z"]
display(df[cols])
print(df.groupby("arm")[["E", "Vscore", "spread_late", "rollbacks"]].median())

## Part 4 — larger-L templates (dual-aware, k = L-1)

Exact-architecture cells, all hyperparameters explicit — duplicate a cell and swap
`arch="GeoCNN", cnn_hidden=...` for the unaware arm when extending the benchmark.
Notes: `chunk_size` is MANDATORY at L>=6 (int32 XLA overflow); `diag_shift=5e-3` at
L>=6 (L=7 primal history); n_sweeps ~ N/3 (N = 3L^3-3L^2: 300/540/882).

In [ ]:
run("scale_dual_L5_hz0.28_s0", dual=True, arch="ToricCNN_gridinv",
    L=5, bc="OBC", hx=0.0, hz=0.28, seed=0,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="4 4", kernel_size=4,               # k = L-1
    n_iter=200, dt=0.02, lr_min=0.002, diag_shift=1e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=100, chunk_size=2048,
    group="scale_L5", subdir="scale")

In [ ]:
run("scale_dual_L6_hz0.26_s0", dual=True, arch="ToricCNN_gridinv",
    L=6, bc="OBC", hx=0.0, hz=0.26, seed=0,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="4 4", kernel_size=5,               # k = L-1
    n_iter=200, dt=0.02, lr_min=0.002, diag_shift=5e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=180, chunk_size=2048,
    group="scale_L6", subdir="scale")

In [ ]:
run("scale_dual_L7_hz0.25_s0", dual=True, arch="ToricCNN_gridinv",
    L=7, bc="OBC", hx=0.0, hz=0.25, seed=0,
    n_noninv=2, noninv_channels=4, radius_edge=1.05,
    inv_hidden="4 4", kernel_size=6,               # k = L-1
    n_iter=175, dt=0.02, lr_min=0.002, diag_shift=5e-3, qgt="dense",
    n_samples=8192, n_chains=1024, n_sweeps=294, chunk_size=1024,
    group="scale_L7", subdir="scale")